# Exploratory Colab experiment

> Cleaned archive of the original graduation-project notebook. For new leakage-aware runs, use the reusable pipeline under src/ and scripts/.


In [ ]:

import os
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# ============================================
# Ortak Parametreler
# ============================================
IMG_SIZE = 299            # InceptionV3 için orijinal giriş boyutu
BATCH_SIZE = 32
NUM_CLASSES = 3           # Sınıf sayınıza göre güncelleyin
INITIAL_LR = 1e-4         # Feature extraction aşaması LR
FINE_TUNE_LR = 1e-5       # Fine-tuning aşaması LR
EPOCHS_FEATURE = 100      # Feature extraction için epoch sayısı
EPOCHS_FINE_TUNE = 20     # Fine-tuning için epoch sayısı
DRIVE_MODEL_PATH = '.'   # Google Drive’daki klasörünüz

# ============================================
# Veri Dizinleri (train, valid, test)
# ============================================
data_root = 'data/split'   # Kendi yolunuza göre ayarlayın
train_dir = os.path.join(data_root, 'train')
valid_dir = os.path.join(data_root, 'valid')
test_dir  = os.path.join(data_root, 'test')         # Eğer test klasörünüz varsa


In [ ]:
# Hücre 2: Temel ayarlar ve ImageDataGenerator’lar

# Görüntü boyutları ve hiperparametreler
IMG_SIZE = 299         # InceptionV3 için standart giriş boyutu 299x299
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 1e-4

# Drive’daki veri yolu (filtreli_veri_1500 klasörünüzün tam yolunu kontrol edin)
BASE_DIR = 'data/split'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VALID_DIR = os.path.join(BASE_DIR, 'test')  # test klasörünü validasyon olarak kullanıyoruz

# 1) Eğitim için data augmentation ve yeniden boyutlandırma
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 2) Validasyon (test) için yalnızca yeniden boyutlandırma
valid_datagen = ImageDataGenerator(
    rescale=1./255
)

# 3) Jeneratörlerin oluşturulması
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Sınıf sayısını dinamik olarak almak (örneğin: 3 sınıf)
num_classes = train_generator.num_classes
print(f"Sınıf sayısı: {num_classes}")
print("Sınıflar ve indexleri:", train_generator.class_indices)

In [ ]:
# 1) İnceptionV3 (include_top=False) yükleme, ImageNet ağırlıkları
base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2) Taban modeli dondurma
for layer in base_model.layers:
    layer.trainable = False

# 3) Üzerine yeni katmanlar ekleme
x = base_model.output
x = GlobalAveragePooling2D(name='GAP')(x)
x = Dropout(0.5, name='dropout_1')(x)  # %50 Dropout
outputs = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

# 4) Modeli tanımlama
model = Model(inputs=base_model.input, outputs=outputs)

# 5) Optimizer ve compile
optimizer = Adam(learning_rate=INITIAL_LR)
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 6) Callback’ler
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1,
    min_lr=1e-7
)
checkpoint_path = os.path.join(DRIVE_MODEL_PATH, 'inceptionv3_best_weights_299.h5')
model_checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks_list = [reduce_lr, model_checkpoint]

# 7) Model özeti (isteğe bağlı)
model.summary()


In [ ]:
history = model.fit(
    train_generator,
    epochs=EPOCHS_FEATURE,
    validation_data=valid_generator,
    callbacks=callbacks_list,
    verbose=1
)


In [ ]:
# Eğitim & Doğrulama Doğruluğu
plt.figure(figsize=(8, 6))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Feature Extraction (299x299) – Eğitim ve Doğrulama Doğruluğu')
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()
plt.grid()
plt.show()

# Eğitim & Doğrulama Kaybı
plt.figure(figsize=(8, 6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Feature Extraction (299x299) – Eğitim ve Doğrulama Kaybı')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# -----------------------------------------------
# Fine‐Tuning: İnce Ayar Aşaması
# -----------------------------------------------

# 1) İnce ayar için son 20 katmanı açalım
for layer in base_model.layers[:249]:
    layer.trainable = False
for layer in base_model.layers[249:]:
    layer.trainable = True

# 2) Modeli düşük öğrenme oranıyla yeniden derleyelim
optimizer_finetune = Adam(learning_rate=FINE_TUNE_LR)
model.compile(
    optimizer=optimizer_finetune,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 3) Fine‐tuning callback’leri (EarlyStopping olmadan)
reduce_lr_finetune = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-8
)
checkpoint_path_finetune = os.path.join(DRIVE_MODEL_PATH, 'inceptionv3_finetuned_best_weights_299.h5')
model_checkpoint_finetune = ModelCheckpoint(
    checkpoint_path_finetune,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks_finetune = [reduce_lr_finetune, model_checkpoint_finetune]

# 4) Fine‐tuning eğitimini başlatalım
FINE_TUNE_EPOCHS = 20

history_finetune = model.fit(
    train_generator,
    epochs=FINE_TUNE_EPOCHS,
    validation_data=valid_generator,
    callbacks=callbacks_finetune,
    verbose=1
)


In [ ]:
import matplotlib.pyplot as plt

# 1) Eskiden elde edilen history ile fine‐tuning history’sini birleştirelim
#    (accuracy, val_accuracy, loss, val_loss list’lerini ardışık şekilde birleştiriyoruz)

# Doğruluk listeleri
combined_train_acc = history.history['accuracy'] + history_finetune.history['accuracy']
combined_val_acc   = history.history['val_accuracy'] + history_finetune.history['val_accuracy']

# Kayıp listeleri
combined_train_loss = history.history['loss'] + history_finetune.history['loss']
combined_val_loss   = history.history['val_loss'] + history_finetune.history['val_loss']

# 2) Toplam epoch sayısını belirleyelim
total_epochs = len(combined_train_acc)  # örn. önce 100 epoch sonra 20 epoch → toplam 120

# 3) Epoch index listesini oluşturalım
epochs = list(range(1, total_epochs + 1))

# 4) Birleştirilmiş Doğruluk Grafiği
plt.figure(figsize=(8, 6))
plt.plot(epochs, combined_train_acc, label='Train Accuracy (Birleştirilmiş)')
plt.plot(epochs, combined_val_acc,   label='Val Accuracy (Birleştirilmiş)')
plt.axvline(x=len(history.history['accuracy']), color='gray', linestyle='--', linewidth=1,
            label='Fine‐tuning Başlangıcı')
plt.title('Train ve Validation Accuracy (Toplam Epoch: {})'.format(total_epochs))
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()
plt.grid()
plt.show()

# 5) Birleştirilmiş Kayıp Grafiği
plt.figure(figsize=(8, 6))
plt.plot(epochs, combined_train_loss, label='Train Loss (Birleştirilmiş)')
plt.plot(epochs, combined_val_loss,   label='Val Loss (Birleştirilmiş)')
plt.axvline(x=len(history.history['loss']), color='gray', linestyle='--', linewidth=1,
            label='Fine‐tuning Başlangıcı')
plt.title('Train ve Validation Loss (Toplam Epoch: {})'.format(total_epochs))
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()
plt.grid()
plt.show()


In [ ]:
import os
import shutil

# 1) HDF5 (.h5) olarak kaydetme örneği
save_path_h5 = 'artifacts/inceptionv3_final.h5'
model.save(save_path_h5)
print(f"Model HDF5 formatında kaydedildi: {save_path_h5}")




In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

# -------------------------------------------------
# A) Önceki fine‐tuning ağırlıklarını yükleyelim
# -------------------------------------------------
# checkpoint_path_finetune, ince‐ayar sırasında en iyi ağırlıkları kaydettiğiniz dosya yolunu göstermeli.
checkpoint_path_finetune = os.path.join(DRIVE_MODEL_PATH, 'inceptionv3_finetuned_best_weights_299.h5')

# load_model, eğer doğrudan model.save() ile kaydettiğiniz .h5 dosyasıysa tam modeli yükler.
# Eğer sadece model.load_weights() ile ağırlıkları sakladıysanız,
# önce aynı mimariyi yeniden oluşturup ardından model.load_weights(checkpoint_path_finetune) demeniz gerekir.

model = load_model(checkpoint_path_finetune)
print("✔ Önceki ince‐ayar ağırlıkları yüklendi:", checkpoint_path_finetune)

# -------------------------------------------------
# B) İnce‐ayar için son katmanları tekrar açalım
# -------------------------------------------------
# (Eğer modeli kaydederken zaten bu ayarı yapmışsanız, burada tekrar yapmanız gerekir.)
for layer in base_model.layers[:249]:
    layer.trainable = False
for layer in base_model.layers[249:]:
    layer.trainable = True

# -------------------------------------------------
# C) Modeli düşük öğrenme oranı ile yeniden derleyelim
# -------------------------------------------------
optimizer_finetune = Adam(learning_rate=FINE_TUNE_LR)
model.compile(
    optimizer=optimizer_finetune,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# -------------------------------------------------
# D) Callback’leri (ReduceLR ve ModelCheckpoint) hazırlayalım
# -------------------------------------------------
reduce_lr_finetune = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-8
)
# Bu defa ağırlıkları “inceptionv3_finetuned_best_weights_299_extra20.h5”
# gibi yeni bir dosyaya kaydedelim ki ilk 20 epoch’un ağırlıkları bozulmasın.
extra_checkpoint_path = os.path.join(DRIVE_MODEL_PATH, 'inceptionv3_finetuned_best_weights_299_extra20.h5')
model_checkpoint_finetune = ModelCheckpoint(
    extra_checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks_finetune = [reduce_lr_finetune, model_checkpoint_finetune]

# -------------------------------------------------
# E) İnce‐ayar eğitiminin “20 epoch daha”sını başlatalım
# -------------------------------------------------
FINE_TUNE_EXTRA = 20  # Ek olarak kaç epoch devam edecekseniz

history_finetune_extra = model.fit(
    train_generator,
    epochs=FINE_TUNE_EXTRA,
    validation_data=valid_generator,
    callbacks=callbacks_finetune,
    verbose=1
)

print("✔ İnce‐ayar: 20 epoch daha tamamlandı. Yeni ağırlıklar kaydedildi:", extra_checkpoint_path)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ----------------------------------------------------------------
# A) History objelerinin birleşmesi (FE + FT1 + FT_EXTRA)
# ----------------------------------------------------------------

# Örnek olarak:
#   history     = feature extraction sonrası history
#   history_finetune      = ilk fine‐tuning sonrası history
#   history_finetune_extra = ek fine‐tuning sonrası history

# 1) Doğruluk listeleri
combined_train_acc = (
    history.history['accuracy']
    + history_finetune.history['accuracy']
    + history_finetune_extra.history['accuracy']
)
combined_val_acc = (
    history.history['val_accuracy']
    + history_finetune.history['val_accuracy']
    + history_finetune_extra.history['val_accuracy']
)

# 2) Kayıp listeleri
combined_train_loss = (
    history.history['loss']
    + history_finetune.history['loss']
    + history_finetune_extra.history['loss']
)
combined_val_loss = (
    history.history['val_loss']
    + history_finetune.history['val_loss']
    + history_finetune_extra.history['val_loss']
)

# 3) Toplam epoch sayısı ve epoch index listesi
total_epochs = len(combined_train_acc)  # örn: 20 + 20 + 20 = 60
epochs = list(range(1, total_epochs + 1))

# 4) Ana aşama başlangıç epoch indeksleri
fe_end   = len(history.history['accuracy'])                 # Feature extraction bitişi
ft1_end  = fe_end + len(history_finetune.history['accuracy'])           # İlk FT bitişi
ft_extra_start = ft1_end  # Ek FT başlangıcı

# ----------------------------------------------------------------
# B) Accuracy & Loss Grafikleri
# ----------------------------------------------------------------

# B1) Accuracy
plt.figure(figsize=(8, 6))
plt.plot(epochs, combined_train_acc, label='Train Accuracy (Combined)')
plt.plot(epochs, combined_val_acc,   label='Val Accuracy (Combined)')

# Dikey çizgiyle aşama geçişlerini gösterelim
plt.axvline(x=fe_end, color='gray', linestyle='--', linewidth=1.2, label=f'FE End (Epoch {fe_end})')
plt.axvline(x=ft1_end, color='black', linestyle='--', linewidth=1.2, label=f'FT1 End (Epoch {ft1_end})')

plt.title(f'Train & Validation Accuracy (Total Epochs: {total_epochs})')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# B2) Loss
plt.figure(figsize=(8, 6))
plt.plot(epochs, combined_train_loss, label='Train Loss (Combined)')
plt.plot(epochs, combined_val_loss,   label='Val Loss (Combined)')

plt.axvline(x=fe_end, color='gray', linestyle='--', linewidth=1.2, label=f'FE End (Epoch {fe_end})')
plt.axvline(x=ft1_end, color='black', linestyle='--', linewidth=1.2, label=f'FT1 End (Epoch {ft1_end})')

plt.title(f'Train & Validation Loss (Total Epochs: {total_epochs})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------
# C) Confusion Matrix ve Classification Report (Validasyon)
# ----------------------------------------------------------------

# 1) En son kaydedilen ince‐ayar ağırlıklarını yükleyelim
#    (Ek fine‐tuning sonrası en iyi ağırlıkları gösteren dosya yolu)
model.load_weights(extra_checkpoint_path)

# 2) Validasyon setindeki gerçek etiketler ve tahminler
y_true = valid_generator.classes
y_pred_probs = model.predict(valid_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# 3) Sınıf etiketlerini al
class_labels = list(valid_generator.class_indices.keys())

# 4) Confusion matrix hesapla
cm = confusion_matrix(y_true, y_pred)
print("=== Confusion Matrix ===")
print(cm)

# 5) Heatmap ile görselleştirme
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=class_labels,
    yticklabels=class_labels,
    cmap='Blues'
)
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title('Confusion Matrix (Validation)')
plt.tight_layout()
plt.show()

# 6) Classification Report
print("\n=== Classification Report ===\n")
print(classification_report(y_true, y_pred, target_names=class_labels))


In [ ]:
import os

FINAL_MODEL_PATH = 'artifacts/inceptionv3_final.h5'

# Eğer eski dosya varsa sil
if os.path.exists(FINAL_MODEL_PATH):
    os.remove(FINAL_MODEL_PATH)

# Yeni modeli kaydet
model.save(FINAL_MODEL_PATH)



In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from tensorflow.keras.models import load_model

# ---------------------------------------------------------
# 0) Önkoşullar:
#
#    a) 'inceptionv3_final.h5' olarak kaydedilen model dosyanızın yolu:
FINAL_MODEL_PATH = 'artifacts/inceptionv3_final.h5'
#
#    b) Aşağıda 'valid_generator' geçerli olduğundan,
#       validasyon verisini sağlayan ImageDataGenerator daha önce tanımlı olmalı.
#       Örnek: valid_generator = valid_datagen.flow_from_directory(VALID_DIR, ...)
#
#    c) valid_generator.class_indices, farklı sınıfların isimlerini (örneğin {'SINIF1':0, 'SINIF2':1, 'SINIF3':2}) verir.
# ---------------------------------------------------------

# 1) Modeli yükleyelim
model = load_model(FINAL_MODEL_PATH)
print("✔ Model başarıyla yüklendi:", FINAL_MODEL_PATH)

# 2) Gerçek etiketleri ve tahmin olasılıklarını elde edelim
#    valid_generator daha önce tanımlanmış olmalı (örneğin ImageDataGenerator ile)
y_true      = valid_generator.classes                 # Shape: (N,)
y_pred_probs = model.predict(valid_generator, verbose=1)  # Shape: (N, num_classes)

# 3) Sınıf isimleri ve ters sözlük
class_indices = valid_generator.class_indices  # Örn: {'SINIF1':0, 'SINIF2':1, 'SINIF3':2}
idx_to_class  = {v: k for k, v in class_indices.items()}

# 4) One-hot formatına çevirme
n_classes = len(class_indices)  # Örneğin 3
y_true_ohe = label_binarize(y_true, classes=list(range(n_classes)))  # (N,3)

# 5) Her sınıf için ROC eğrisi ve AUC hesaplayalım
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_ohe[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# 6) Mikro ve makro ortalama (isteğe bağlı)
# Mikro-average
fpr["micro"], tpr["micro"], _ = roc_curve(y_true_ohe.ravel(), y_pred_probs.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Makro-average
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# 7) ROC Eğrilerini Çizelim
plt.figure(figsize=(8, 6))

# Her sınıf için ayrı çizim
colors = ["darkorange", "green", "blue"]  # Sınıf sayısına göre seçebilirsiniz
for i, color in zip(range(n_classes), colors):
    plt.plot(
        fpr[i],
        tpr[i],
        color=color,
        lw=2,
        label=f"ROC (class = {idx_to_class[i]}) (AUC = {roc_auc[i]:.2f})"
    )

# Mikro ve makro ortalama eğrileri
plt.plot(
    fpr["micro"],
    tpr["micro"],
    color="deeppink",
    linestyle=":",
    linewidth=2,
    label=f"micro-average ROC (AUC = {roc_auc['micro']:.2f})"
)
plt.plot(
    fpr["macro"],
    tpr["macro"],
    color="navy",
    linestyle=":",
    linewidth=2,
    label=f"macro-average ROC (AUC = {roc_auc['macro']:.2f})"
)

# Diyagonal referans çizgisi (rastgele sınıflandırıcı)
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.title("Multi-class ROC Eğrisi")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

# ----- 0. Sınıf isimlerini (target_names) oluştur -----
# test_generator.class_indices: {'tip1': 0, 'tip2': 1, 'tip3': 2, ...} gibi bir sözlük döner.
# Biz bu sözlüğü indeks sırasına göre sıralayıp bir listeye koyacağız:
#
# Örneğin:
#   test_generator.class_indices = {'tip2': 1, 'tip1': 0, 'tip3': 2}
#   sorted(test_generator.class_indices.items(), key=lambda x: x[1])
#   → [('tip1', 0), ('tip2', 1), ('tip3', 2)]
#
# Böylece indeks sırasına göre isimleri elde etmiş oluyoruz.

target_names = [class_name for class_name, _ in
                sorted(test_generator.class_indices.items(), key=lambda x: x[1])]

# ----- 1. Rastgele bir test resmi seç -----
file_paths = test_generator.filepaths
random_index = random.randint(0, len(file_paths) - 1)
img_path = file_paths[random_index]

# ----- 2. Görüntüyü yükle ve normalize et -----
# IMG_SIZE: Daha önce tanımlanmış olmalı (örneğin 224, 299 vb.).
# Eğer IMG_SIZE kullanılmıyorsa, IMG_HEIGHT ve IMG_WIDTH kullanacak şekilde ayarla.
img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# ----- 3. Model ile tahmin yap ve sınıf indekslerini al -----
pred_probs = model.predict(img_array)[0]        # Örneğin: [0.10, 0.70, 0.20]
predicted_idx = np.argmax(pred_probs)
predicted_class = target_names[predicted_idx]

true_idx = test_generator.classes[random_index]
true_class = target_names[true_idx]

# ----- 4. Sonuçları ekrana bastır -----
print(f"Gerçek:   {true_class}")
print(f"Tahmin:   {predicted_class}")
print()  # boş bir satır
for i, name in enumerate(target_names):
    print(f"{name}: {pred_probs[i]:.2f}")

# ----- 5. Alt kısımda yalnızca resmi göster -----
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.show()